In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

# Go 3 levels up from notebook location to project root
PROJECT_ROOT = Path.cwd().parents[2]

RESULTS_DIR = PROJECT_ROOT / "results"
EXTERNAL_DATA_DIR = PROJECT_ROOT / "data" / "external"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("EXTERNAL_DATA_DIR exists:", EXTERNAL_DATA_DIR.exists())

STRATEGIES = {
    "BTP_Italia": "btp_italia/index_daily.csv",
    "iTraxx_Combined": "itraxx_combined/index_daily.csv",
    "CDS_Bond_Basis": "cds_bond_basis/index_daily.csv",
}

VARIANTS = {
    "EW": "index_return_ew",
    "SW": "index_return_sw",
}


PROJECT_ROOT: c:\Users\aless\Desktop\THESIS
EXTERNAL_DATA_DIR exists: True


In [5]:
euribor_path = EXTERNAL_DATA_DIR / "Euribor1m.xlsx"

euribor_df = pd.read_excel(
    euribor_path,
    skiprows=7,
    header=None,
    usecols=[0, 1],
    names=["Date", "Euribor1M"],
    parse_dates=["Date"],
)

euribor_df.set_index("Date", inplace=True)
euribor_df = euribor_df.dropna()

euribor_df["rf_daily_pct"] = euribor_df["Euribor1M"] * (1 / 360)
rf_daily_dec = euribor_df["rf_daily_pct"] / 100.0

print("RF range:", rf_daily_dec.index.min(), "->", rf_daily_dec.index.max())


RF range: 2004-01-02 00:00:00 -> 2026-02-17 00:00:00


In [6]:
for name, rel_path in STRATEGIES.items():
    print("\n" + "="*90)
    print(f"STRATEGY: {name}")
    print("="*90)

    full_path = RESULTS_DIR / rel_path
    print("CSV:", full_path)
    df = pd.read_csv(full_path, index_col=0, parse_dates=True)

    print("Date range:", df.index.min(), "->", df.index.max())
    print("Rows:", len(df))
    print("Columns:", list(df.columns))

    for vname, col in VARIANTS.items():
        print(f"\n--- {vname} ({col}) ---")

        if col not in df.columns:
            print("❌ Column NOT found.")
            continue

        s_raw = df[col]
        print("raw dtype:", s_raw.dtype)
        print("raw non-null:", s_raw.notna().sum())

        s_num = pd.to_numeric(s_raw, errors="coerce")
        print("numeric non-null:", s_num.notna().sum())

        s_dec = s_num / 100.0
        rf_on_dates = rf_daily_dec.reindex(s_dec.index)
        aligned = pd.DataFrame({"r": s_dec, "rf": rf_on_dates}).dropna()

        print("RF non-null on these dates:", rf_on_dates.notna().sum())
        print("Aligned rows (r & rf):", len(aligned))

        if len(aligned) > 0:
            print("Aligned date range:", aligned.index.min(), "->", aligned.index.max())
            print("Mean r (daily, dec):", aligned["r"].mean())



STRATEGY: BTP_Italia
CSV: c:\Users\aless\Desktop\THESIS\results\btp_italia\index_daily.csv
Date range: 2012-04-03 00:00:00 -> 2025-10-17 00:00:00
Rows: 3389
Columns: ['index_return', 'cumulative_return', 'index_value', 'index_return_sw', 'cumulative_return_sw', 'index_value_sw', 'index_return_ew', 'cumulative_return_ew', 'index_value_ew']

--- EW (index_return_ew) ---
raw dtype: float64
raw non-null: 3282
numeric non-null: 3282
RF non-null on these dates: 3389
Aligned rows (r & rf): 3282
Aligned date range: 2012-04-03 00:00:00 -> 2025-10-17 00:00:00
Mean r (daily, dec): 6.900732696083314e-05

--- SW (index_return_sw) ---
raw dtype: float64
raw non-null: 3282
numeric non-null: 3282
RF non-null on these dates: 3389
Aligned rows (r & rf): 3282
Aligned date range: 2012-04-03 00:00:00 -> 2025-10-17 00:00:00
Mean r (daily, dec): 7.883901096972569e-05

STRATEGY: iTraxx_Combined
CSV: c:\Users\aless\Desktop\THESIS\results\itraxx_combined\index_daily.csv
Date range: 2005-09-21 00:00:00 -> 2025-